# Task 6 · SFT only
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
checkout = Path(CHECKOUT)
if not checkout.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
subprocess.run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(checkout / PROJECT_SUBDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 1. Install the pinned GPU training environment
After this cell, restart the runtime if Colab requests it, then rerun the clone/setup cell.
Do not install the CPU-only requirements in vendor/organizer into this GPU environment.


In [ ]:
%pip -q install -r requirements.txt
!python -m pip check
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Copy data from Drive or download directly from Hugging Face
All extraction and image reads happen on local SSD, not mounted Drive.


In [ ]:
import shutil
USE_DRIVE_DATA = True
drive_raw = Path('/content/drive/MyDrive/MMCQA/task6-data/raw')
if USE_DRIVE_DATA:
    if not drive_raw.exists():
        raise FileNotFoundError('Run the transfer notebook first, or set USE_DRIVE_DATA=False')
    shutil.copytree(drive_raw, Path('data/raw'), dirs_exist_ok=True)
else:
    subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'download'], check=True)
subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'prepare'], check=True)


## 3. Editable experiment settings
Edit prompts/qa.txt and YAML files, or use dotted --set overrides below.
4-bit is explicit for a memory-limited Colab GPU. Set False for native precision.
GPU smoke runs are needed before full training; they have not been executed on the CPU-only development machine.


In [ ]:
USE_4BIT = True
RUN_FULL = False
RUN_NAME = 'experiment01'  # use a new name for a fresh experiment
RUNS = Path('/content/drive/MyDrive/MMCQA/task6-runs') / RUN_NAME
RUNS.mkdir(parents=True, exist_ok=True)
SFT = str(RUNS / ('sft' if RUN_FULL else 'sft-smoke'))
GRPO = str(RUNS / ('grpo' if RUN_FULL else 'grpo-smoke'))

def run(module, config, overrides):
    cmd = [sys.executable, '-m', module, '--config', config]
    for key, value in overrides.items():
        cmd += ['--set', f'{key}={value}']
    subprocess.run(cmd, check=True)

common = {'model.load_in_4bit': str(USE_4BIT).lower()}


## 4. SFT (answer-only loss)
Smoke: 1 optimizer step. Full: configured 2 epochs, internal validation used for checkpoint selection.
Saved `final` contains LoRA adapter and processor; the base weights are not duplicated.


In [ ]:
sft_args = {**common, 'training.output_dir': SFT}
if not RUN_FULL:
    sft_args.update({'data.limit': 8, 'training.max_steps': 1, 'training.eval_strategy': 'no',
                     'training.load_best_model_at_end': 'false', 'training.save_steps': 1})
run('mmcultureqa.training.sft', 'configs/sft.yaml', sft_args)


## 5. SFT checkpoint on Drive
Training writes periodic checkpoints and the final adapter directly to Drive.
`final` contains adapter + processor; base Qwen weights are downloaded separately.
Copy the printed path into SFT_ADAPTER in notebook 03. You may end this Colab session.
For SFT resume, set resume_from_checkpoint to a checkpoint-N directory in the SFT run cell.


In [ ]:
SFT_FINAL = Path(SFT) / 'final'
if not (SFT_FINAL / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'SFT did not produce a final adapter: {SFT_FINAL}')
print('SFT adapter saved on Drive:', SFT_FINAL)
print('Use this path as SFT_ADAPTER in notebook 03.')
